Loading and preprocessing the dataset

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Hyperparameters
batch_size = 64
learning_rate = 0.001
epochs = 10

# Transform: convert to tensor and normalize
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Load datasets
train_dataset = datasets.FashionMNIST(
    root="./data", train=True, transform=transform, download=True
)
test_dataset = datasets.FashionMNIST(
    root="./data", train=False, transform=transform, download=True
)

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

100%|██████████| 26.4M/26.4M [00:01<00:00, 14.6MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 272kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.01MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 8.73MB/s]


Constructing Deep Feedforward Neural Network

In [ ]:
class DeepFNN(nn.Module):
    def __init__(self):
        super(DeepFNN, self).__init__()
        self.fc1 = nn.Linear(28*28, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(x.size(0), -1)  # Flatten
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.fc4(x)  # Linear output (logits)
        return x


Loss Function and Optimizer

In [ ]:
model = DeepFNN().to(device)

criterion = nn.CrossEntropyLoss()   # Softmax is included internally
optimizer = optim.Adam(model.parameters(), lr=learning_rate)


Training Loop

In [ ]:
def train(model, loader):
    model.train()
    correct, total, running_loss = 0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    return running_loss / len(loader), correct / total


Evaluation on the dataset

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    return correct / total


Accuracy

In [ ]:
for epoch in range(epochs):
    train_loss, train_acc = train(model, train_loader)
    test_acc = evaluate(model, test_loader)

    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Loss: {train_loss:.4f} "
          f"Train Acc: {train_acc:.4f} "
          f"Test Acc: {test_acc:.4f}")


Epoch [1/10] Loss: 0.5015 Train Acc: 0.8165 Test Acc: 0.8288
Epoch [2/10] Loss: 0.3703 Train Acc: 0.8638 Test Acc: 0.8514
Epoch [3/10] Loss: 0.3322 Train Acc: 0.8771 Test Acc: 0.8711
Epoch [4/10] Loss: 0.3065 Train Acc: 0.8863 Test Acc: 0.8719
Epoch [5/10] Loss: 0.2856 Train Acc: 0.8940 Test Acc: 0.8703
Epoch [6/10] Loss: 0.2690 Train Acc: 0.8994 Test Acc: 0.8695
Epoch [7/10] Loss: 0.2541 Train Acc: 0.9047 Test Acc: 0.8760
Epoch [8/10] Loss: 0.2422 Train Acc: 0.9092 Test Acc: 0.8800
Epoch [9/10] Loss: 0.2305 Train Acc: 0.9141 Test Acc: 0.8842
Epoch [10/10] Loss: 0.2188 Train Acc: 0.9167 Test Acc: 0.8828
